[Guide Index](README.md)


# 03 SPARQL

`OxGraph` runs SPARQL 1.2 on Oxigraph's own engine. (`OxGraph` is the working name of what becomes `StarLayerGraph`; see the [Oxigraph wrapper guide](02d-oxigraph-wrapper.ipynb).) The query text is sent to Oxigraph as written, so triple-term-aware functions, quoted-triple patterns in queries and updates, Turtle 1.2-based shorthand syntax and direction-tagged literals all work without any rewriting by StarLayer.

See the [Graphs guide](02-graphs.ipynb) for the triple terms, base-direction literals and reification used in the examples below. Terms in query results are pyoxigraph terms (`NamedNode`, `BlankNode`, `Literal`, `Triple`); they have no `.n3()` method, so the examples print them with `g.qname_term(term)` (prefixed names, `_:` labels) or `str(term)`.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
from starlayer.graph.oxgraph import OxGraph

EX = "http://example.org/"
RDF = "http://www.w3.org/1999/02/22-rdf-syntax-ns#"

## 1. Query semantics and built-in functions

`OxGraph.query()` supports SPARQL 1.2 query execution:
- Query over reified quoted triples
- Turtle 1.2 quoted-triple syntax in queries (`<<( ... )>>`)
- Binds variables for terms inside triple terms (`?s ?p ?o`)

A new `OxGraph` has no prefixes. The graph below is built with `ex` and `rdf` bound, and the graph passes its prefixes to Oxigraph on every `query()` and `serialize()`, so the queries do not need `PREFIX` lines for them (a `PREFIX` line in the query text still overrides). The examples run against the example graph, created here.

In [2]:
g_parsed = OxGraph(prefixes={"ex": EX, "rdf": RDF})
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals: AcmeCorp's name in English and Arabic
    ex:AcmeCorp ex:name "Acme Corporation"@en--ltr .
    ex:AcmeCorp ex:name "شركة أكمي"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> ;
      ex:source ex:HRSystem ;
      ex:confidence "high" .

    # anonymous inline annotation block (asserted triple)
    ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} .

    # named reifier with annotations
    ex:alice ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} .

    # named reifier without annotation block
    ex:alice ex:worksWith ex:SalesTeam ~ ex:stmt2 .
    ex:stmt2 ex:asReported ex:HRSystem .
    ex:stmt2 ex:reportedDate "1/1/2026" .

    # reification shorthand << s p o >> (no parens): an anonymous, unasserted
    # reifier of the same base triple ex:claim reifies above - a triple
    # can have more than one independent reification
    << ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:LinkedIn .

    # unasserted quoted triple term
    ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
''', format='turtle')
print("parsed triples:", len(g_parsed))
print()
# pretty=True is StarLayer's pretty Turtle writer (grouped subjects, {| |} annotation blocks);
# the default serialize() is Oxigraph's flat one-statement-per-line Turtle.
print(g_parsed.serialize(format="turtle", pretty=True))

parsed triples: 20

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:AcmeCorp ex:name "Acme Corporation"@en--ltr, "شركة أكمي"@ar--rtl .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} ;
    ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} ;
    ex:worksWith ex:SalesTeam ~ ex:stmt2 {| ex:asReported ex:HRSystem ; ex:reportedDate "1/1/2026" |} .

<< ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim >> ex:confidence "high" ;
    ex:source ex:HRSystem .

<< ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:LinkedIn .



In [3]:
# SPARQL query to bind all terms inside a quoted triple
rows = g_parsed.query("""
SELECT ?claim ?s ?p ?o ?source WHERE {
  ?claim rdf:reifies <<( ?s ?p ?o )>> .
  ?claim ex:source ?source .
  FILTER(?p = ex:worksFor)
}
ORDER BY ?claim ?s ?o
""")

# ?claim can be an anonymous reifier (a BlankNode) as well as a named one (a NamedNode);
# qname_term() writes either: a prefixed name for an IRI, _:label for a blank node.
for row in rows:
    print(
        g_parsed.qname_term(row.claim),
        g_parsed.qname_term(row.s),
        g_parsed.qname_term(row.p),
        g_parsed.qname_term(row.o),
        "Source: ", g_parsed.qname_term(row.source),
    )

_:a68de5ac717963043fa8bf3557ec8cb3 ex:alice ex:worksFor ex:AcmeCorp Source:  ex:LinkedIn
ex:claim ex:alice ex:worksFor ex:AcmeCorp Source:  ex:HRSystem


In [4]:
# detect triple-term values using isTRIPLE
rows = g_parsed.query("""
SELECT ?claim ?p WHERE {
  ?claim ?p ?statement .
  FILTER( isTRIPLE(?statement) )
}
ORDER BY ?claim
""")

for row in rows:
    print(g_parsed.qname_term(row.claim), g_parsed.qname_term(row.p))

_:a68de5ac717963043fa8bf3557ec8cb3 rdf:reifies
_:c5c8b6f377780aacc96f664a00e9a9f3 rdf:reifies
ex:AuditSystem ex:reported
ex:claim rdf:reifies
ex:stmt1 rdf:reifies
ex:stmt2 rdf:reifies


### 1.1 Additional SPARQL 1.2 functions

- `TRIPLE(s, p, o)` — a functional way to write a triple term `<<( s p o )>>`
- `SUBJECT()`, `PREDICATE()`, `OBJECT()` — extract the parts out of a triple term
- `LANGDIR()`, `hasLANGDIR()`, `STRLANGDIR()` — read, check, and build a literal's base direction
- `LANG()`, `hasLANG()` — SPARQL 1.1 functions, now also aware of direction-tagged literals

In [5]:
# 1.1.1 TRIPLE() and SUBJECT()/PREDICATE()/OBJECT()
print("1.1.1 TRIPLE()/SUBJECT()/PREDICATE()/OBJECT()")
rows = g_parsed.query("""
SELECT ?s ?p ?o WHERE {
  ex:claim rdf:reifies ?t .
  FILTER(?t = TRIPLE(ex:alice, ex:worksFor, ex:AcmeCorp))
  BIND(SUBJECT(?t) AS ?s)
  BIND(PREDICATE(?t) AS ?p)
  BIND(OBJECT(?t) AS ?o)
}
""")
for row in rows:
    print(g_parsed.qname_term(row.s), g_parsed.qname_term(row.p), g_parsed.qname_term(row.o))

# 1.1.2 The <<( s p o )>> literal syntax works directly in BIND() too - equivalent
# to TRIPLE(s, p, o) above, not just a WHERE-clause pattern form.
print()
print("1.1.2 <<( s p o )>> in BIND()")
rows = g_parsed.query("""
SELECT ?tt WHERE {
  BIND( <<( ex:alice ex:worksFor ex:AcmeCorp )>> AS ?tt )
}
""")
for row in rows:
    # a pyoxigraph Triple: a triple term is a value of its own, with .subject/.predicate/.object
    print(type(row.tt).__name__, row.tt)

1.1.1 TRIPLE()/SUBJECT()/PREDICATE()/OBJECT()
ex:alice ex:worksFor ex:AcmeCorp

1.1.2 <<( s p o )>> in BIND()
Triple <http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp>


In [6]:
# 1.1.3 LANGDIR() / hasLANGDIR() / LANG() / hasLANG() over AcmeCorp's two names
print("1.1.3 LANGDIR()/hasLANGDIR()/LANG()/hasLANG()")
rows = g_parsed.query("""
SELECT ?s ?lang ?dir ?hasDir WHERE {
  ?s ex:name ?lit .
  BIND(LANG(?lit) AS ?lang)
  BIND(LANGDIR(?lit) AS ?dir)
  BIND(hasLANGDIR(?lit) AS ?hasDir)
}
ORDER BY ?s ?lang
""")
for row in rows:
    # row.lang and row.dir are Literals; .value is the Python str
    print(g_parsed.qname_term(row.s), row.lang.value, row.dir.value, row.hasDir.value)

# 1.1.4 STRLANGDIR() constructs a direction-tagged literal directly from plain strings
print()
print("1.1.4 STRLANGDIR()")
rows = g_parsed.query('SELECT ?lit ' \
'WHERE { BIND(STRLANGDIR("hi", "en", "ltr") AS ?lit) }')
for row in rows:
    print(row.lit, "| language:", row.lit.language, "| direction:", row.lit.direction)

1.1.3 LANGDIR()/hasLANGDIR()/LANG()/hasLANG()
ex:AcmeCorp ar rtl true
ex:AcmeCorp en ltr true

1.1.4 STRLANGDIR()
"hi"@en--ltr | language: en | direction: ltr


### 1.2 Turtle annotation shorthand inside SPARQL queries

The `{| ?pred ?val |}`, `~ ?r`, and `<< s p o >>` Turtle syntax used to parse and serialize graphs also works directly inside a SPARQL 1.2 `WHERE` clause. This allows querying using the same shorthand a graph is serialized with, without expanding to `rdf:reifies`/`<<( )>>`.

In [7]:
# 1.2.1 {| ?pred ?val |}: query an anonymous reifier's annotations inline (asserted triple)
print("1.2.1 {| ?pred ?val |}")
rows = g_parsed.query("""
SELECT ?pred ?val WHERE {
  ex:alice ex:likes ex:ProductABC {| ?pred ?val |}
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname_term(row.pred), g_parsed.qname_term(row.val))

# 1.2.2 ~ ?r: bind the reifier itself for a named reifier
print()
print("1.2.2 ~ ?r")
rows = g_parsed.query("""
SELECT ?r WHERE {
  ex:alice ex:worksWith ex:SalesTeam ~ ?r
}
""")
for row in rows:
    print(g_parsed.qname_term(row.r))

# 1.2.3 << s p o >> ?pred ?val: reification shorthand, no assertion required
print()
print("1.2.3 << s p o >>")
rows = g_parsed.query("""
SELECT ?pred ?val WHERE {
  << ex:alice ex:likes ex:ProductABC >> ?pred ?val

}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname_term(row.pred), g_parsed.qname_term(row.val))

1.2.1 {| ?pred ?val |}
ex:since "2020"
ex:source ex:CRM

1.2.2 ~ ?r
ex:stmt2

1.2.3 << s p o >>
ex:since "2020"
ex:source ex:CRM
rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>>


### 1.3 The `VERSION "1.2"` query prologue directive

SPARQL 1.2 introduces an optional version number, allowing a query to document which version of SPARQL it is using.
A leading `VERSION "1.2"` line (before any `PREFIX`/`SELECT`) declares that a query uses SPARQL 1.2 features. Declaring a version that doesn't match what the query actually uses (e.g. `VERSION "1.1"` on a query containing a triple term) emits a `SPARQL12ConformanceWarning`.

In [8]:
import warnings

# 1.3.1 VERSION "1.2" declared, and the query does use RDF 1.2 syntax - no warning.
# ?stmt can match an anonymous reifier (a BlankNode) as well as a named one,
# so print via qname_term(), which handles either.
print("1.3.1 VERSION \"1.2\" matches usage")
rows = g_parsed.query("""
VERSION "1.2"
SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
""")
print(sorted(g_parsed.qname_term(row.stmt) for row in rows))

# 1.3.2 VERSION "1.1" declared, but the query text itself uses a triple term - warns.
# The warning comes from StarLayer's check before the text is sent; Oxigraph itself runs the query.
print()
print("1.3.2 VERSION \"1.1\" mismatch warns")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    rows = g_parsed.query("""
    VERSION "1.1"
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
    """)
    list(rows)
    for w in caught:
        print(f"{w.category.__name__}: {w.message}")

1.3.1 VERSION "1.2" matches usage
['_:a68de5ac717963043fa8bf3557ec8cb3', 'ex:claim']

1.3.2 VERSION "1.1" mismatch warns


## 2. SPARQL 1.2 Update

`INSERT DATA`/`DELETE DATA` allow for the use of SPARQL 1.2 syntax, including triple-term patterns.



In [9]:
g_update = OxGraph(prefixes={"ex": EX, "rdf": RDF})

# INSERT DATA with a ground triple term - the base triple is not asserted.
g_update.update("""
    INSERT DATA { ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
""")
print("triples:", len(g_update))
print()
print(g_update.serialize(format="turtle"))

triples: 1

@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



In [10]:
# INSERT ... WHERE: a triple-term pattern with variables, matched first,
# then instantiated into the INSERT template.
g_update.update("""
    INSERT { ?stmt ex:confidence "high" }
    WHERE  { ?stmt rdf:reifies <<( ?s ex:worksFor ?o )>> }
""")
print(g_update.serialize(format="turtle"))

@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:claim ex:confidence "high" ;
	rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

